# Lab 18-09 · Current Budgets, Sleep Modes and Battery Pulses

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import ulp

## Part 1 · Budget

*Predict* Example 18-09.1's average current and life with advertising every 2 s.

In [2]:
for adv_period in (1.0, 2.0):
    i = ulp.average_current([(3e-3, 2e-3), (6e-3, 3e-3 / adv_period)], 1.0, 1.5e-6)
    print(f"advertising every {adv_period:.0f} s: {i * 1e6:.2f} uA, {ulp.battery_life_hours(220, i) / 24:.0f} days")
q = ulp.simulate_day([(1.0, 0.0, 3e-3, 2e-3), (1.0, 0.002, 6e-3, 3e-3)], 1.5e-6)
print(f"simulated day, advertising every 1 s: average {q / 86400 * 1e6:.3f} uA")

advertising every 1 s: 25.49 uA, 360 days
advertising every 2 s: 16.49 uA, 556 days
simulated day, advertising every 1 s: average 25.492 uA


## Part 2 · Modes

*Predict* the idle times at which stop beats sleep and shutdown beats stop.

In [3]:
t = np.logspace(-5, 2, 400)
modes = {"sleep": (0.8e-3, 0.0), "stop": (3e-6, 100e-9), "shutdown": (0.1e-6, 100e-9 + 4e-6)}
for name, (i, q) in modes.items():
    plt.loglog(t, i * t + q, label=name)
plt.xlabel("idle time (s)"); plt.ylabel("charge per idle period (C)"); plt.legend(); plt.show()
print("crossovers:", ulp.break_even_time(100e-9, 0.8e-3, 3e-6), "s and", ulp.break_even_time(4e-6, 3e-6, 0.1e-6), "s")

crossovers: 0.00012547051442910915 s and 1.379310344827586 s


## Part 3 · Race

*Predict* how the charge per second varies with clock, and where the best point is when a 1.2 V range is available up to 24 MHz.

In [4]:
f = np.linspace(2e6, 48e6, 200)
q18 = [ulp.charge_per_task(x, 1e6, 60e-12, 400e-6, 2e-6, 1.0) for x in f]
f12 = f[f <= 24e6]
q12 = [ulp.charge_per_task(x, 1e6, 60e-12 * 1.2 / 1.8, 400e-6 * 1.2 / 1.8, 2e-6, 1.0) for x in f12]
plt.plot(f / 1e6, np.array(q18) * 1e6, label="1.8 V"); plt.plot(f12 / 1e6, np.array(q12) * 1e6, label="1.2 V range")
plt.xlabel("clock (MHz)"); plt.ylabel("charge per second (uC)"); plt.legend(); plt.show()
print(f"48 MHz at 1.8 V: {q18[-1] * 1e6:.2f} uC; 24 MHz at 1.2 V: {q12[-1] * 1e6:.2f} uC")

48 MHz at 1.8 V: 70.29 uC; 24 MHz at 1.2 V: 53.05 uC


## Part 4 · The burst

*Predict* the droop with 47, 100 and 220 µF.

In [5]:
for c in (47e-6, 100e-6, 220e-6):
    print(f"{c * 1e6:5.0f} uF: {ulp.droop_with_cap(8e-3, 100, c, 2e-3) * 1e3:6.1f} mV")
if shutil.which("ngspice"):
    out = subprocess.run(["ngspice", "-b", "coin_cell_pulse.cir"], capture_output=True, text=True).stdout
    print("ngspice:", re.search(r"droop_cap_mv\s*=\s*(\S+)", out).group(1), "mV with 100 uF;",
          re.search(r"droop_nocap_v\s*=\s*(\S+)", out).group(1), "V without")

   47 uF:  277.3 mV
  100 uF:  145.0 mV
  220 uF:   69.5 mV
ngspice: 1.45015e+02 mV with 100 uF; 8.00000e-01 V without


## Part 5 · Self-discharge

*Predict* where self-discharge dominates.

In [6]:
for i_avg in (2e-6, 20e-6, 200e-6):
    lives = [ulp.battery_life_hours(1000, i_avg, 1.0, sd) / 8760 for sd in (0.0, 0.01 / 12, 0.03)]
    print(f"{i_avg * 1e6:5.0f} uA: " + ", ".join(f"{x:6.2f} y" for x in lives) + "   (none, 1%/year, 3%/month)")

    2 uA:  57.08 y,  36.15 y,   2.61 y   (none, 1%/year, 3%/month)
   20 uA:   5.71 y,   5.40 y,   1.85 y   (none, 1%/year, 3%/month)
  200 uA:   0.57 y,   0.57 y,   0.47 y   (none, 1%/year, 3%/month)


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).